In [ ]:
import os

folder_name = "Downloaded_Data"
folder_path = rf"C:\INCOIS\ARGO\DATA\{folder_name}"

os.makedirs(folder_path, exist_ok=True)

print("Folder created:", folder_path)

In [ ]:
import pandas as pd

file_path = r"C:\INCOiS\ARGO\DATA\Raw_Data\ar_index_global_prof.txt"

df = pd.read_csv(file_path, comment="#")

print(df.head())

In [ ]:
print(df.columns)

In [ ]:
date=df['date'].values
print(date)
longitude = df['longitude'].values
print(longitude)
latitude=df['latitude'].values
print(latitude)
institution=df['institution'].values
print(institution)
ocean=df['ocean'].values
print(ocean)

In [ ]:

ocean_df=df[df['ocean']=='I']
print(ocean_df.head())
print(ocean_df['ocean'])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
!pip install cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Map boundaries
lon_min, lon_max = 20, 120
lat_min, lat_max = -70,30

g_size=int(input("enter size of grid"))
rows=(lat_max-lat_min)/g_size
cols=(lon_max-lon_min)/g_size
rows=int(rows)
cols=int(cols)

# Create grid lines
lon_lines = np.linspace(lon_min, lon_max, cols + 1)
lat_lines = np.linspace(lat_min, lat_max, rows + 1)

# Create figure
fig = plt.figure(figsize=(15,10))

# Create map projection
ax = plt.axes(projection=ccrs.PlateCarree())

# Set map extent
ax.set_extent([lon_min, lon_max, lat_min, lat_max])

# Add ocean and land
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.LAND)

# Add coastlines
ax.coastlines()

# Draw vertical grid lines
for lon in lon_lines:
    ax.plot(
        [lon, lon],
        [lat_min, lat_max],
        color='black',
        transform=ccrs.PlateCarree()
    )

# Draw horizontal grid lines
for lat in lat_lines:
    ax.plot(
        [lon_min, lon_max],
        [lat, lat],
        color='black',
        transform=ccrs.PlateCarree()
    )

# Add grid IDs
grid_id = 1

for i in reversed(range(rows)):
    for j in range(cols):

        # Grid center
        x_center = (lon_lines[j] + lon_lines[j+1]) / 2
        y_center = (lat_lines[i] + lat_lines[i+1]) / 2

        # Add text label
        ax.text(
            x_center,
            y_center,
            f"{grid_id}",
            transform=ccrs.PlateCarree(),
            ha='center',
            fontsize=10,
            color='black'
        )

        grid_id += 1
ax.scatter(
    ocean_df['longitude'],
    ocean_df['latitude'],
    s=0.5,

    color='lightcoral',
    transform=ccrs.PlateCarree()   # ✅ Correct
)

# Add title
plt.title("Ocean Basemap with 5x5 Grid IDs")

# Show map
plt.show()

In [ ]:
import numpy as np
ocean_df = ocean_df.copy()
# Grid size
g_size = 5

ocean_df['grid_col'] = (
    ((ocean_df['longitude'] - lon_min) // g_size).astype(int)

)

ocean_df['grid_row'] = (
    ((lat_max - ocean_df['latitude']) // g_size).astype(int)

)
ocean_df['grid_id'] = (
    ocean_df['grid_row'] * cols
    + ocean_df['grid_col']
    + 1)

grid7 = ocean_df[ocean_df['grid_id'] == 7]

print(grid7)
grid_profiles = dict(tuple(ocean_df.groupby('grid_id')))
profile_count = ocean_df.groupby('grid_id').size()

print(profile_count)
grid7 = ocean_df[ocean_df['grid_id'] == 7]

print(profile_count.head(40))

In [ ]:
grid4= ocean_df[ocean_df['grid_id'] == 4]

print(len(grid4))

In [ ]:
print(grid4['file'].iloc[0])

In [ ]:
base_url =  "https://data-argo.ifremer.fr/dac/"

file_path = grid4['file'].iloc[0]

url = base_url + file_path

print(url)

In [ ]:
import requests

url =  "https://data-argo.ifremer.fr/dac/" + grid4['file'].iloc[0]

r = requests.get(url)

with open("profile.nc", "wb") as f:
    f.write(r.content)

print("Downloaded")

In [ ]:
import os
import requests

base_url = "https://data-argo.ifremer.fr/dac/"
file_path = grid4['file'].iloc[0]

url = base_url + file_path

# Local folder
save_folder = r"C:\INCOiS\ARGO\DATA"
os.makedirs(save_folder, exist_ok=True)

# Extract filename from URL
filename = os.path.basename(file_path)

# Full local path
local_file = os.path.join(save_folder, filename)

print("Downloading:")
print(url)

response = requests.get(url, stream=True)
response.raise_for_status()

with open(local_file, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)

print(f"Downloaded successfully: {local_file}")

In [ ]:
import requests
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

# Local destination
root_folder = r"C:\INCOiS\ARGO\DATA\South_Data"
os.makedirs(root_folder, exist_ok=True)

def download_file(url, save_path):

    # Skip if already downloaded
    if os.path.exists(save_path):
        return f"Skipped {os.path.basename(save_path)}"

    try:
        response = requests.get(
            url,
            stream=True,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=120
        )

        response.raise_for_status()

        with open(save_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        return f"✔ {os.path.basename(save_path)}"

    except Exception as e:
        return f"✖ {os.path.basename(save_path)} : {e}"


futures = []

# Increase cautiously
MAX_WORKERS = 40

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    for grid_id, grid_data in south_df.groupby("grid_id"):

        grid_folder = os.path.join(
            root_folder,
            f"grid_{grid_id}"
        )

        os.makedirs(grid_folder, exist_ok=True)

        print(
            f"Grid {grid_id}: {len(grid_data)} files"
        )

        # ALL files in grid
        for file_path in grid_data["file"]:

            url = (
                "https://data-argo.ifremer.fr/dac/"
                + file_path
            )

            filename = os.path.basename(file_path)

            save_path = os.path.join(
                grid_folder,
                filename
            )

            futures.append(
                executor.submit(
                    download_file,
                    url,
                    save_path
                )
            )

    completed = 0

    for future in as_completed(futures):
        completed += 1

        if completed % 100 == 0:
            print(
                f"Completed {completed}/{len(futures)}"
            )

print("\nAll downloads finished.")

In [ ]:
import os

root_folder = r"C:\INCOiS\ARGO\DATA\South_Data"

downloaded = 0

for root, dirs, files in os.walk(root_folder):
    downloaded += len([f for f in files if f.endswith(".nc")])

print("Downloaded files:", downloaded)

In [ ]:
expected = south_df["file"].nunique()

print("Expected files :", expected)
print("Downloaded files:", downloaded)
print("Remaining:", expected - downloaded)
print("Completion:",
      round(downloaded/expected*100,2), "%")

In [ ]:
import os
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

# ==========================================================
# Root folder
# ==========================================================
root_folder = r"C:\INCOiS\ARGO\DATA\South_Data"

# ==========================================================
# Download function
# ==========================================================

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

def download_file(url, save_path):

    try:
        response = session.get(url, stream=True, timeout=120)
        response.raise_for_status()

        with open(save_path, "wb") as f:
            for chunk in response.iter_content(1024*1024):
                if chunk:
                    f.write(chunk)

        return f"Downloaded: {os.path.basename(save_path)}"

    except Exception as e:
        return f"Failed: {os.path.basename(save_path)} ({e})"

# ==========================================================
# Find Missing Files
# ==========================================================

missing = []

for grid_id, grid_data in south_df.groupby("grid_id"):

    grid_folder = os.path.join(root_folder, f"grid_{grid_id}")

    os.makedirs(grid_folder, exist_ok=True)

    for file_path in grid_data["file"].drop_duplicates():

        filename = os.path.basename(file_path)

        save_path = os.path.join(grid_folder, filename)

        if not os.path.exists(save_path):

            url = "https://data-argo.ifremer.fr/dac/" + file_path

            missing.append((url, save_path))

print(f"\nMissing files found: {len(missing)}")

# ==========================================================
# Download only Missing Files
# ==========================================================

MAX_WORKERS = 32

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    futures = [
        executor.submit(download_file, url, save_path)
        for url, save_path in missing
    ]

    completed = 0

    for future in as_completed(futures):

        completed += 1

        if completed % 100 == 0 or completed == len(missing):
            print(f"{completed}/{len(missing)} completed")

print("\nAll missing files downloaded.")

In [ ]:
from netCDF4 import Dataset
import pandas as pd
import numpy as np
import os
import gc

base_folder = r"C:\INCOiS\ARGO\DATA\South_Data"

grid_folders = sorted(
    [d for d in os.listdir(base_folder) if d.startswith("grid_")],
    key=lambda x: int(x.split("_")[1])
)

# Start from Grid 195 onwards
grid_folders = [
    g for g in grid_folders
    if int(g.split("_")[1]) >= 219
]

print("Grids to process:")
print(grid_folders)

output_folder = os.path.join(base_folder, "Grid_Parquet")
os.makedirs(output_folder, exist_ok=True)

for grid_folder_name in grid_folders:

    rows = []      # Fresh list for every grid

    grid_id = int(grid_folder_name.split("_")[1])

    grid_folder = os.path.join(base_folder, grid_folder_name)

    nc_files = [
        f for f in os.listdir(grid_folder)
        if f.endswith(".nc")
    ]

    print(f"\n========== GRID {grid_id} ==========")
    print(f"Profiles : {len(nc_files)}")

    for count, file in enumerate(nc_files, start=1):

        file_path = os.path.join(
            grid_folder,
            file
        )

        try:

            ds = Dataset(file_path)

            lat = float(ds.variables["LATITUDE"][:][0])
            lon = float(ds.variables["LONGITUDE"][:][0])

            try:
                date = "".join(
                    ds.variables["REFERENCE_DATE_TIME"][:]
                    .astype(str)
                ).strip()
            except:
                date = ""

            pres = np.ma.filled(
                ds.variables["PRES"][:],
                np.nan
            ).flatten()

            temp = np.ma.filled(
                ds.variables["TEMP"][:],
                np.nan
            ).flatten()

            psal = np.ma.filled(
                ds.variables["PSAL"][:],
                np.nan
            ).flatten()

            temp_qc = np.ma.filled(
                ds.variables["TEMP_QC"][:],
                np.nan
            ).flatten()

            psal_qc = np.ma.filled(
                ds.variables["PSAL_QC"][:],
                np.nan
            ).flatten()

            # Make arrays same length
            n = min(len(pres), len(temp), len(psal))
            
            pres = pres[:n]
            temp = temp[:n]
            psal = psal[:n]
            
            valid = (
                ~np.isnan(pres)
                & ~np.isnan(temp)
                & ~np.isnan(psal)
            )
            
            pres = pres[valid]
            temp = temp[valid]
            psal = psal[valid]
            
            if len(temp_qc) >= n:
                temp_qc = temp_qc[:n][valid]
            
            if len(psal_qc) >= n:
                psal_qc = psal_qc[:n][valid]

            if len(temp) == 0:
                ds.close()
                continue
            print(
              f"Grid {grid_id} | {file} | "
              f"Valid records = {len(temp)}")

            temp_min = np.min(temp)
            temp_max = np.max(temp)
            temp_mean = np.mean(temp)

            psal_min = np.min(psal)
            psal_max = np.max(psal)
            psal_mean = np.mean(psal)

            pres_mean = np.mean(pres)

            for i in range(len(temp)):

                rows.append({

                    "GRID_ID": grid_id,
                    "PROFILE": file,
                    "DATE": date,

                    "LON": lon,
                    "LAT": lat,

                    "TEMP_MIN": temp_min,
                    "TEMP_MAX": temp_max,
                    "TEMP_MEAN": temp_mean,

                    "PSAL_MIN": psal_min,
                    "PSAL_MAX": psal_max,
                    "PSAL_MEAN": psal_mean,

                    "PRES_MEAN": pres_mean,

                    "PRES": float(pres[i]),
                    "TEMP": float(temp[i]),
                    "PSAL": float(psal[i]),

                    "TEMP_QC": (
                        str(temp_qc[i])
                        if i < len(temp_qc)
                        else ""
                    ),

                    "PSAL_QC": (
                        str(psal_qc[i])
                        if i < len(psal_qc)
                        else ""
                    )
                })

            ds.close()

            if count % 200 == 0:
                print(f"Processed {count}/{len(nc_files)}")

        except Exception as e:
            print(f"Error in {file}: {e}")

    # ==============================
    # Save current grid
    # ==============================

    print("Creating DataFrame...")

    df = pd.DataFrame(rows)

    print("Rows:", len(df))

    output_file = os.path.join(
        output_folder,
        f"grid_{grid_id}.parquet"
    )

    df.to_parquet(output_file, index=False)

    print(f"Saved: {output_file}")

    # Free memory before next grid
    del df
    del rows
    gc.collect()

print("\nAll grids completed.")

In [3]:
import os
import pandas as pd

# Folder containing all grid parquet files
parquet_folder = r"C:\INCOiS\ARGO\DATA\South_Data\Grid_Parquet"

# Output file
output_file = r"C:\INCOiS\ARGO\DATA\South_Data\south.parquet"

# Get all parquet files
parquet_files = sorted(
    [
        os.path.join(parquet_folder, f)
        for f in os.listdir(parquet_folder)
        if f.endswith(".parquet")
    ]
)

print(f"Found {len(parquet_files)} parquet files")

dfs = []

for i, file in enumerate(parquet_files, start=1):

    print(f"Reading {i}/{len(parquet_files)} : {os.path.basename(file)}")

    df = pd.read_parquet(file)

    dfs.append(df)

print("\nCombining all files...")

final_df = pd.concat(dfs, ignore_index=True)

print("Total Rows:", len(final_df))
print("Total Columns:", len(final_df.columns))

print("\nSaving final parquet...")

final_df.to_parquet(output_file, index=False)

print("\nSaved Successfully!")
print(output_file)

Found 143 parquet files
Reading 1/143 : grid_112.parquet
Reading 2/143 : grid_113.parquet
Reading 3/143 : grid_114.parquet
Reading 4/143 : grid_115.parquet
Reading 5/143 : grid_116.parquet
Reading 6/143 : grid_132.parquet
Reading 7/143 : grid_133.parquet
Reading 8/143 : grid_134.parquet
Reading 9/143 : grid_135.parquet
Reading 10/143 : grid_136.parquet
Reading 11/143 : grid_137.parquet
Reading 12/143 : grid_152.parquet
Reading 13/143 : grid_153.parquet
Reading 14/143 : grid_154.parquet
Reading 15/143 : grid_155.parquet
Reading 16/143 : grid_156.parquet
Reading 17/143 : grid_157.parquet
Reading 18/143 : grid_158.parquet
Reading 19/143 : grid_159.parquet
Reading 20/143 : grid_160.parquet
Reading 21/143 : grid_172.parquet
Reading 22/143 : grid_173.parquet
Reading 23/143 : grid_174.parquet
Reading 24/143 : grid_175.parquet
Reading 25/143 : grid_176.parquet
Reading 26/143 : grid_177.parquet
Reading 27/143 : grid_178.parquet
Reading 28/143 : grid_179.parquet
Reading 29/143 : grid_180.parquet

In [11]:
import pyarrow.parquet as pq

# Open parquet file
parquet_file = pq.ParquetFile(
    r"C:\INCOiS\ARGO\DATA\South_Data\south.parquet"
)

# Read only the first row group
table = parquet_file.read_row_group(0)

# Convert to pandas DataFrame
df = table.to_pandas()

# Display
print(df.shape)
display(df.head(10))

(1048576, 17)


,GRID_ID,PROFILE,DATE,LON,LAT,TEMP_MIN,TEMP_MAX,TEMP_MEAN,PSAL_MIN,PSAL_MAX,PSAL_MEAN,PRES_MEAN,PRES,TEMP,PSAL,TEMP_QC,PSAL_QC
0,112,D1900158_060.nc,19500101000000,76.907997,0.21,6.749,28.624001,15.074176,34.948002,35.519001,35.161316,315.841217,4.700000,28.613001,35.466999,b'1',b'1'
1,112,D1900158_060.nc,19500101000000,76.907997,0.21,6.749,28.624001,15.074176,34.948002,35.519001,35.161316,315.841217,9.200000,28.621000,35.473999,b'1',b'1'
2,112,D1900158_060.nc,19500101000000,76.907997,0.21,6.749,28.624001,15.074176,34.948002,35.519001,35.161316,315.841217,19.200001,28.624001,35.476002,b'1',b'1'
3,112,D1900158_060.nc,19500101000000,76.907997,0.21,6.749,28.624001,15.074176,34.948002,35.519001,35.161316,315.841217,29.400000,28.544001,35.519001,b'1',b'1'
4,112,D1900158_060.nc,19500101000000,76.907997,0.21,6.749,28.624001,15.074176,34.948002,35.519001,35.161316,315.841217,39.400002,28.488001,35.504002,b'1',b'1'
5,112,D1900158_060.nc,19500101000000,76.907997,0.21,6.749,28.624001,15.074176,34.948002,35.519001,35.161316,315.841217,49.400002,28.316000,35.487000,b'1',b'1'
6,112,D1900158_060.nc,19500101000000,76.907997,0.21,6.749,28.624001,15.074176,34.948002,35.519001,35.161316,315.841217,59.900002,27.990999,35.476002,b'1',b'1'
7,112,D1900158_060.nc,19500101000000,76.907997,0.21,6.749,28.624001,15.074176,34.948002,35.519001,35.161316,315.841217,69.300003,27.002001,35.442001,b'1',b'1'
8,112,D1900158_060.nc,19500101000000,76.907997,0.21,6.749,28.624001,15.074176,34.948002,35.519001,35.161316,315.841217,79.699997,24.486000,35.382999,b'1',b'1'
9,112,D1900158_060.nc,19500101000000,76.907997,0.21,6.749,28.624001,15.074176,34.948002,35.519001,35.161316,315.841217,89.599998,22.993000,35.424000,b'1',b'1'
